In [0]:
%python
#validando variaveis 
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
bronze_mapeamento= {
    'temp_bronze_clientes' : f'{bronze_path}/clientes/',
    'temp_bronze_itens_pedido' : f'{bronze_path}/itens_pedido/',
    'temp_bronze_pedidos' : f'{bronze_path}/pedidos/',
    'temp_bronze_produtos' : f'{bronze_path}/produtos/',
    'temp_bronze_vendedores' : f'{bronze_path}/vendedores/'

}
for view_name, path in bronze_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)

)

In [0]:
%sql
--consultando tabelas temporarias
select * from temp_bronze_vendedores

In [0]:
%sql
select * from temp_bronze_itens_pedido

In [0]:
%sql
select * from temp_bronze_pedidos

In [0]:
%sql
select * from temp_bronze_produtos

In [0]:
%sql
select * from temp_bronze_clientes

In [0]:
%sql
describe temp_bronze_clientes

In [0]:
%python
#tratamento dos dados 
df_silver_cliente = spark.sql("""
    SELECT
        id_cliente,
        nome,
        email,
        estado,
        DATE_FORMAT(data_cadastro, 'dd/MM/yyyy') AS data_cadastro,
        status AS status_cliente

    FROM temp_bronze_clientes

    WHERE LOWER(TRIM(status)) <> 'inativo'

    GROUP BY
        id_cliente,
        nome,
        email,
        estado,
        DATE_FORMAT(data_cadastro, 'dd/MM/yyyy'),
        status
""")


# Salvar em delta na silver
df_silver_cliente.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{silver_path}/clientes')

In [0]:
%python
#consultando tabela tratada conforme a regra de negocio
display(df_silver_cliente)